# Module 04 — ML Theory & Evaluation
## 04-09: Calibration & Uncertainty Quantification

**Objective:** Implement reliability diagrams, ECE, temperature scaling, Platt scaling,
isotonic regression, and conformal prediction — learning when a model's confidence is
trustworthy and how to fix it when it is not.

**Prerequisites:** 4-01 (Evaluation Metrics), 1-07 (Information Theory Basics)


## Part 0 — Setup & Prerequisites

A model's **predicted probability** $\hat{p}$ is *well-calibrated* if it matches the
true empirical frequency: if the model says 0.8 for 100 examples, roughly 80 of them
should be positive.  Miscalibration is common — neural networks tend to be *overconfident*;
Naive Bayes tends to push probabilities toward 0 and 1.

We implement and compare five calibration approaches:

| Method | Parameters | Flexibility | Typical Use |
|--------|------------|-------------|-------------|
| Reliability diagram + ECE | 0 (diagnostic) | — | Measuring miscalibration |
| Temperature scaling | 1 ($T$) | Low | Fast post-hoc fix for NNs / LLMs |
| Platt scaling | 2 ($a$, $b$) | Low | Binary classifiers (SVMs) |
| Isotonic regression | $n$ (non-parametric) | High | Moderate calibration set |
| Conformal prediction | 1 (threshold) | Distribution-free | Guaranteed coverage sets |

**Prerequisites:** 4-01 (Evaluation Metrics), 1-07 (Information Theory Basics)

> **Theory notebook:** every method is implemented from scratch and validated empirically.


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys
import math
import warnings
warnings.filterwarnings("ignore")
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits, make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.isotonic import IsotonicRegression as SklearnIsotonic
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from scipy.optimize import minimize_scalar

import torch

import random
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"NumPy: {np.__version__}")
if torch.cuda.is_available():
    print(f"CUDA: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# ── Reproducibility ───────────────────────────────────────────────────────────

SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Seed: {SEED}")


In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
N_BINS        = 10       # bins for reliability diagrams
ALPHA_CP      = 0.10     # conformal prediction target miscoverage (90% coverage)
N_SYNTH       = 3000     # synthetic dataset size
TRAIN_FRAC    = 0.60     # 60% train / 20% calibration / 20% test
CAL_FRAC      = 0.50     # split non-train equally into cal + test
TEMP_BOUNDS   = (0.05, 30.0)   # temperature search range
TEMP_TOL      = 1e-9           # convergence tolerance for T optimisation

print("Configuration:")
print(f"  N_BINS     = {N_BINS}  |  ALPHA_CP  = {ALPHA_CP} (target {1-ALPHA_CP:.0%} coverage)")
print(f"  N_SYNTH    = {N_SYNTH}  |  TRAIN_FRAC = {TRAIN_FRAC}")
print(f"  CAL_FRAC   = {CAL_FRAC}  (each of cal/test = {(1-TRAIN_FRAC)/2:.2f} of data)")


### Data Generation

We use **Gaussian Naive Bayes** (GNB) as our intentionally miscalibrated classifier.
GNB assumes features are conditionally independent, which is rarely true — this causes it
to push probabilities toward 0 and 1 (overconfidence).  We hold out a *calibration set*
(separate from the test set) for all post-hoc calibration methods.


In [ ]:
# ── Synthetic Binary Classification: 3-way Split for Calibration ──────────────
X_synth, y_synth = make_classification(
    n_samples=N_SYNTH, n_features=20, n_informative=10, n_redundant=5,
    flip_y=0.05, class_sep=0.8, random_state=SEED,
)

X_tr, X_rest, y_tr, y_rest = train_test_split(
    X_synth, y_synth, test_size=1.0 - TRAIN_FRAC, random_state=SEED,
)
X_cal, X_te, y_cal, y_te = train_test_split(
    X_rest, y_rest, test_size=CAL_FRAC, random_state=SEED + 1,
)

# Train GaussianNB — known to be overconfident on correlated features
gnb_synth = GaussianNB()
gnb_synth.fit(X_tr, y_tr)

# Raw probability matrices (n x 2)
p_raw_cal_2d = gnb_synth.predict_proba(X_cal)
p_raw_te_2d  = gnb_synth.predict_proba(X_te)
p_raw_cal    = p_raw_cal_2d[:, 1]   # positive-class prob, shape (n_cal,)
p_raw_te     = p_raw_te_2d[:, 1]    # positive-class prob, shape (n_te,)

acc_gnb = float(accuracy_score(y_te, gnb_synth.predict(X_te)))
print("Synthetic binary classification dataset:")
print(f"  Train: {len(X_tr):4d} | Cal: {len(X_cal):4d} | Test: {len(X_te):4d}")
print(f"  Class balance (train positive rate): {y_tr.mean():.3f}")
print(f"  GNB test accuracy                  : {acc_gnb:.4f}")
print(f"  Mean P(Y=1) on test (should ~ {y_te.mean():.3f}): {p_raw_te.mean():.4f}")
print(f"  Prob range: [{p_raw_te.min():.4f}, {p_raw_te.max():.4f}]  std={p_raw_te.std():.4f}")


---
## Part 1 — Theoretical Foundation & Implementation

### 1.1 Calibration Metrics: Reliability Diagram and ECE

A classifier is **perfectly calibrated** if, for every confidence level $p$:
$$
P(Y = 1 \mid \hat{p}(X) = p) = p
$$

**Reliability diagram:** bin predictions by confidence; compare mean confidence $\overline{\hat{p}}_b$
against fraction of positives in each bin.  Perfect calibration = all bars on the diagonal.

**Expected Calibration Error (ECE):**
$$
\text{ECE} = \sum_{b=1}^{B} \frac{|\mathcal{B}_b|}{n}
              \left| \text{acc}(\mathcal{B}_b) - \text{conf}(\mathcal{B}_b) \right|
$$

**Maximum Calibration Error (MCE)** = $\max_b |\text{acc}(b) - \text{conf}(b)|$ (worst bin).

**Multi-class ECE (confidence calibration):** use $\max_k \hat{p}_k$ as confidence and
$\mathbf{1}[\hat{y} = y]$ as accuracy.


In [ ]:
# ── Part 1: Reliability Diagram and Calibration Error ─────────────────────────


def reliability_diagram_data(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    n_bins: int = 10,
) -> dict[str, Any]:
    '''Compute reliability diagram bin statistics for binary predictions.

    Divides [0, 1] into n_bins equal-width intervals, and for each bin computes
    the mean confidence and fraction of positive samples.

    Args:
        y_true: Binary ground-truth labels (0 or 1), shape (n,).
        y_prob: Predicted probability of class 1, shape (n,).
        n_bins: Number of equal-width bins.

    Returns:
        Dict with keys: bin_centers, accuracies, confidences, fractions, counts.
    '''
    edges       = np.linspace(0.0, 1.0, n_bins + 1)
    bin_centers = 0.5 * (edges[:-1] + edges[1:])
    accuracies  = np.zeros(n_bins)
    confidences = np.zeros(n_bins)
    counts      = np.zeros(n_bins, dtype=int)

    for b in range(n_bins):
        if b < n_bins - 1:
            mask = (y_prob >= edges[b]) & (y_prob < edges[b + 1])
        else:
            mask = (y_prob >= edges[b]) & (y_prob <= edges[b + 1])
        n_b = int(mask.sum())
        if n_b > 0:
            accuracies[b]  = float(y_true[mask].mean())
            confidences[b] = float(y_prob[mask].mean())
            counts[b]      = n_b

    n_total   = max(int(counts.sum()), 1)
    fractions = counts.astype(float) / n_total
    return {
        "bin_centers": bin_centers,
        "accuracies":  accuracies,
        "confidences": confidences,
        "fractions":   fractions,
        "counts":      counts,
    }


def expected_calibration_error(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    n_bins: int = 10,
) -> tuple[float, float]:
    '''Compute Expected Calibration Error (ECE) and Maximum Calibration Error (MCE).

    ECE = sum_b (|B_b|/n) * |acc(b) - conf(b)|
    MCE = max_b |acc(b) - conf(b)|

    Args:
        y_true: Binary ground-truth labels (0 or 1), shape (n,).
        y_prob: Predicted probability of class 1, shape (n,).
        n_bins: Number of equal-width bins.

    Returns:
        Tuple (ECE, MCE) as floats.
    '''
    d    = reliability_diagram_data(y_true, y_prob, n_bins)
    gaps = np.abs(d["accuracies"] - d["confidences"])
    ece  = float(np.sum(d["fractions"] * gaps))
    mce  = float(gaps.max()) if len(gaps) > 0 else 0.0
    return ece, mce


def multi_class_ece(
    y_true: np.ndarray,
    p_matrix: np.ndarray,
    n_bins: int = 10,
) -> float:
    '''Compute ECE for multi-class classification via confidence calibration.

    Uses max-probability as the confidence and correctness as the accuracy target.

    Args:
        y_true: Integer class labels, shape (n,).
        p_matrix: Probability matrix, shape (n, n_classes).
        n_bins: Number of bins.

    Returns:
        ECE as float.
    '''
    conf_max  = p_matrix.max(axis=1)
    pred_cls  = p_matrix.argmax(axis=1)
    correct   = (pred_cls == y_true).astype(float)
    ece_mc, _ = expected_calibration_error(correct, conf_max, n_bins)
    return ece_mc


def plot_reliability_diagram(
    diag_data: dict[str, Any],
    ax: Any,
    title: str = "Reliability Diagram",
    bar_color: str = "steelblue",
    ece: float | None = None,
) -> None:
    '''Plot a reliability diagram on the given matplotlib axes.

    Args:
        diag_data: Output of reliability_diagram_data().
        ax: Matplotlib axes to plot on.
        title: Plot title (ECE is appended if provided).
        bar_color: Colour for the accuracy bars.
        ece: Optional ECE value to append to the title.
    '''
    centers    = diag_data["bin_centers"]
    accuracies = diag_data["accuracies"]
    fractions  = diag_data["fractions"]
    width      = centers[1] - centers[0] if len(centers) > 1 else 0.1

    ax.plot([0, 1], [0, 1], "k--", lw=1.5, label="Perfect calibration", zorder=3)
    for i, (c, a, f) in enumerate(zip(centers, accuracies, fractions)):
        if f > 0:
            ax.bar(c, a, width=width * 0.9, align="center",
                   color=bar_color, alpha=0.72, edgecolor="white", lw=0.5)
            ax.bar(c, max(0.0, c - a), bottom=a, width=width * 0.9,
                   align="center", color="tomato", alpha=0.3)

    ece_tag = f"  ECE={ece:.4f}" if ece is not None else ""
    ax.set_title(f"{title}{ece_tag}", fontsize=10, fontweight="bold")
    ax.set_xlabel("Confidence")
    ax.set_ylabel("Accuracy")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8)


print("reliability_diagram_data | expected_calibration_error | multi_class_ece | plot_reliability_diagram defined.")
ece_demo, mce_demo = expected_calibration_error(y_te, p_raw_te, N_BINS)
print(f"Uncalibrated GNB (test): ECE={ece_demo:.5f}  MCE={mce_demo:.5f}")


In [ ]:
# ── Visualise Uncalibrated GNB Predictions ─────────────────────────────────────
ece_raw, mce_raw = expected_calibration_error(y_te, p_raw_te, N_BINS)
diag_raw = reliability_diagram_data(y_te, p_raw_te, N_BINS)

fig_raw, axes_raw = plt.subplots(1, 2, figsize=(13, 5))

plot_reliability_diagram(diag_raw, axes_raw[0],
                          title="Gaussian NB (uncalibrated)",
                          bar_color="steelblue", ece=ece_raw)

axes_raw[1].hist(p_raw_te[y_te == 0], bins=30, alpha=0.6, color="tomato",    label="Negative (y=0)")
axes_raw[1].hist(p_raw_te[y_te == 1], bins=30, alpha=0.6, color="steelblue", label="Positive (y=1)")
axes_raw[1].axvline(0.5, color="black", ls="--", lw=1.5, label="Threshold 0.5")
axes_raw[1].set_xlabel("Predicted probability P(Y=1)")
axes_raw[1].set_ylabel("Count")
axes_raw[1].set_title("GNB Probability Distribution by True Class",
                       fontsize=10, fontweight="bold")
axes_raw[1].legend(fontsize=9)

plt.suptitle("Gaussian NB — Before Calibration", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"GNB miscalibration summary:")
print(f"  ECE = {ece_raw:.5f}  MCE = {mce_raw:.5f}")
print(f"  Bins:")
hdr_conf = "Conf"; hdr_acc = "Acc"; hdr_gap = "Gap"; hdr_n = "Count"; hdr_dir = "Direction"
print(f"  {'Bin':>6s}  {hdr_conf:>8s}  {hdr_acc:>8s}  {hdr_gap:>8s}  {hdr_n:>6s}  {hdr_dir}")
print("  " + "-" * 55)
for i in range(N_BINS):
    if diag_raw["counts"][i] > 0:
        conf_i = diag_raw["confidences"][i]
        acc_i  = diag_raw["accuracies"][i]
        gap_i  = conf_i - acc_i
        direction = "overconf" if gap_i > 0 else "underconf"
        print(f"  [{diag_raw['bin_centers'][i]:.2f}]  {conf_i:8.4f}  {acc_i:8.4f}  {abs(gap_i):8.4f}  {diag_raw['counts'][i]:6d}  {direction}")
print(f"  GNB is predominantly overconfident — probabilities cluster near 0 and 1.")


### 1.2 Temperature Scaling

Temperature scaling divides logits (or log-odds) by a single scalar $T > 0$ before applying
sigmoid/softmax:
$$
\hat{p}_T = \sigma\!\left(\frac{\text{logit}(\hat{p})}{T}\right),
\quad \text{logit}(p) = \log\frac{p}{1-p}
$$

- $T > 1$: **softens** probabilities toward 0.5 — fixes **overconfidence**.
- $T < 1$: **sharpens** probabilities toward 0 or 1 — fixes **underconfidence**.
- $T = 1$: identity.

$T$ is found by minimising the **negative log-likelihood** on a held-out calibration set.
Predictions (argmax) are unchanged — only confidence values are affected.
This single-parameter method is the standard post-hoc calibration for neural networks,
and the same mechanism used to control LLM generation temperature (Modules 17–18).


In [ ]:
# ── Part 1: Temperature Scaling ────────────────────────────────────────────────


def temperature_scale_binary(
    p_raw: np.ndarray,
    T: float,
) -> np.ndarray:
    '''Apply temperature scaling to binary predicted probabilities.

    Computes sigmoid(logit(p_raw) / T).

    Args:
        p_raw: Uncalibrated probabilities P(Y=1), shape (n,).
        T: Temperature parameter.  T > 1 softens, T < 1 sharpens.

    Returns:
        Calibrated probabilities, shape (n,).
    '''
    p_clipped = np.clip(p_raw, 1e-7, 1.0 - 1e-7)
    logits    = np.log(p_clipped) - np.log(1.0 - p_clipped)
    scaled    = logits / max(float(T), 1e-6)
    return 1.0 / (1.0 + np.exp(-scaled))


def temperature_scale_multiclass(
    p_matrix: np.ndarray,
    T: float,
) -> np.ndarray:
    '''Apply temperature scaling to a multi-class probability matrix.

    Scales log-probabilities by 1/T and renormalises via softmax.

    Args:
        p_matrix: Raw probability matrix, shape (n, n_classes). Rows sum to 1.
        T: Temperature parameter.

    Returns:
        Calibrated probability matrix, shape (n, n_classes).
    '''
    p_clipped = np.clip(p_matrix, 1e-9, 1.0)
    log_p     = np.log(p_clipped)
    scaled    = log_p / max(float(T), 1e-6)
    # Numerically stable softmax
    shifted   = scaled - scaled.max(axis=1, keepdims=True)
    exp_vals  = np.exp(shifted)
    return exp_vals / exp_vals.sum(axis=1, keepdims=True)


class TemperatureScaler:
    '''Post-hoc calibration by optimising a single temperature parameter T.

    Finds T that minimises the negative log-likelihood on a held-out calibration
    set.  Works for both binary (1-D probabilities) and multi-class (2-D matrix).

    Attributes:
        T: Learned temperature value.
        multiclass: True for multi-class problems.
    '''

    def __init__(self, multiclass: bool = False) -> None:
        '''Initialise TemperatureScaler with T=1.0 and given multiclass mode.
                Args:
        multiclass: If True, use multi-class temperature scaling.
        '''
        self.T: float = 1.0
        self.multiclass: bool = multiclass

    def _binary_nll(
        self,
        T: float,
        p_raw: np.ndarray,
        y: np.ndarray,
    ) -> float:
        '''Binary NLL at temperature T.

        Args:
            T: Temperature candidate.
            p_raw: Raw probabilities, shape (n,).
            y: Binary labels, shape (n,).

        Returns:
            Mean negative log-likelihood.
        '''
        p_cal = np.clip(temperature_scale_binary(p_raw, T), 1e-7, 1.0 - 1e-7)
        return -float(np.mean(y * np.log(p_cal) + (1.0 - y) * np.log(1.0 - p_cal)))

    def _mc_nll(
        self,
        T: float,
        p_matrix: np.ndarray,
        y: np.ndarray,
    ) -> float:
        '''Multi-class NLL at temperature T.

        Args:
            T: Temperature candidate.
            p_matrix: Raw probability matrix, shape (n, n_classes).
            y: Integer class labels, shape (n,).

        Returns:
            Mean negative log-likelihood.
        '''
        p_cal = temperature_scale_multiclass(p_matrix, T)
        n     = len(y)
        return -float(np.mean(np.log(p_cal[np.arange(n), y] + 1e-9)))

    def fit(
        self,
        y_cal: np.ndarray,
        p_cal: np.ndarray,
    ) -> "TemperatureScaler":
        '''Optimise T by minimising NLL on calibration data.

        Args:
            y_cal: True labels on calibration set.
            p_cal: Raw probabilities. Shape (n,) binary or (n, C) multi-class.

        Returns:
            Self (fitted scaler).
        '''
        if self.multiclass:
            obj = lambda T: self._mc_nll(T, p_cal, y_cal)
        else:
            obj = lambda T: self._binary_nll(T, p_cal, y_cal)
        result  = minimize_scalar(obj, bounds=TEMP_BOUNDS, method="bounded",
                                  options={"xatol": TEMP_TOL})
        self.T  = float(result.x)
        return self

    def calibrate(self, p_raw: np.ndarray) -> np.ndarray:
        '''Apply calibration with the learned temperature.

        Args:
            p_raw: Raw probabilities. Shape (n,) binary or (n, C) multi-class.

        Returns:
            Calibrated probabilities, same shape as input.
        '''
        if self.multiclass:
            return temperature_scale_multiclass(p_raw, self.T)
        return temperature_scale_binary(p_raw, self.T)


print("TemperatureScaler defined.")
# Demonstrate the effect of different temperatures on probability transformation
T_vals_demo = [0.5, 1.0, 2.0, 5.0]
p_demo      = np.array([0.55, 0.70, 0.80, 0.90, 0.95, 0.99])
print("Temperature scaling: P_raw -> P_calibrated")
header_col = "P_raw"
print(f"  {header_col:>8s}", end="")
for T_d in T_vals_demo:
    header_t = f"T={T_d}"
    print(f"  {header_t:>8s}", end="")
print()
for p_v in p_demo:
    print(f"  {p_v:8.2f}", end="")
    for T_d in T_vals_demo:
        p_scaled = float(temperature_scale_binary(np.array([p_v]), T_d)[0])
        print(f"  {p_scaled:8.4f}", end="")
    print()


In [ ]:
# ── Visualise Temperature Scaling Transformation ───────────────────────────────
p_range = np.linspace(0.01, 0.99, 200)

fig_tv, axes_tv = plt.subplots(1, 2, figsize=(13, 5))
T_palette = ["#d62728", "#ff7f0e", "#2ca02c", "#1f77b4", "#9467bd"]

# Left: calibration mapping P_raw -> P_calibrated for various T
for T_val, col_t in zip([0.3, 0.7, 1.0, 2.0, 5.0], T_palette):
    p_cal_curve = temperature_scale_binary(p_range, T_val)
    ls_t        = "-" if T_val == 1.0 else "--"
    lw_t        = 2.0 if T_val == 1.0 else 1.5
    axes_tv[0].plot(p_range, p_cal_curve, ls=ls_t, lw=lw_t, color=col_t,
                    label=f"T={T_val}")
axes_tv[0].plot([0, 1], [0, 1], "k:", lw=1, label="T=1 (identity)")
axes_tv[0].set_xlabel("Raw P(Y=1)")
axes_tv[0].set_ylabel("Calibrated P(Y=1)")
axes_tv[0].set_title("Temperature Scaling Transformation", fontsize=11, fontweight="bold")
axes_tv[0].legend(fontsize=9)
axes_tv[0].set_xlim(0, 1); axes_tv[0].set_ylim(0, 1)

# Right: NLL vs T curve for synthetic GNB calibration data
T_sweep  = np.linspace(0.1, 10.0, 150)
nll_vals = []
ts_for_viz = TemperatureScaler(multiclass=False)
for T_s in T_sweep:
    nll_vals.append(ts_for_viz._binary_nll(T_s, p_raw_cal, y_cal))
nll_arr = np.array(nll_vals)

ts_synth_viz = TemperatureScaler(multiclass=False).fit(y_cal, p_raw_cal)
T_opt = ts_synth_viz.T
nll_opt = ts_for_viz._binary_nll(T_opt, p_raw_cal, y_cal)

axes_tv[1].plot(T_sweep, nll_arr, "steelblue", lw=2)
axes_tv[1].axvline(T_opt, color="red", ls="--", lw=2, label=f"T*={T_opt:.3f}")
axes_tv[1].scatter([T_opt], [nll_opt], c="red", s=80, zorder=5)
axes_tv[1].axvline(1.0, color="gray", ls=":", lw=1.5, label="T=1 (no scaling)")
axes_tv[1].set_xlabel("Temperature T")
axes_tv[1].set_ylabel("NLL on calibration set")
axes_tv[1].set_title("NLL vs Temperature: optimal T > 1 (GNB overconfident)",
                      fontsize=11, fontweight="bold")
axes_tv[1].legend(fontsize=9)

plt.suptitle("Temperature Scaling: Transformation Curve and NLL Objective",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Optimal temperature T* = {T_opt:.4f}  (T > 1 confirms GNB is overconfident)")
print(f"NLL at T=1 (no scaling): {ts_for_viz._binary_nll(1.0, p_raw_cal, y_cal):.5f}")
print(f"NLL at T*              : {nll_opt:.5f}  (lower = better calibration)")


### 1.3 Platt Scaling

**Platt scaling** fits a logistic regression on the model's raw log-odds, learning
two parameters $a$ and $b$:
$$
\hat{p}_{\text{Platt}}(x) = \sigma(a \cdot \text{logit}(\hat{p}(x)) + b)
$$

Parameters are learned by maximising the likelihood on a calibration set.
Originally designed for SVMs (Platt 1999), it works for any monotone score output.
More expressive than temperature scaling (shifts the probability mapping, not just scales it).


In [ ]:
# ── Part 1: Platt Scaling ──────────────────────────────────────────────────────


class PlattScaler:
    '''Two-parameter logistic calibration on log-odds features (Platt 1999).

    Fits logistic regression on logit(p_raw): sigma(a*logit(p) + b).

    Attributes:
        a: Slope of the logistic fit (learned).
        b: Intercept of the logistic fit (learned).
    '''

    def __init__(self) -> None:
        '''Initialise PlattScaler with identity mapping (a=1, b=0).
                Attributes are overwritten by fit().
        '''
        self.a: float = 1.0
        self.b: float = 0.0
        self._logreg: LogisticRegression = LogisticRegression(
            C=1e10, fit_intercept=True, max_iter=1000,
        )

    def fit(
        self,
        y_cal: np.ndarray,
        p_cal: np.ndarray,
    ) -> "PlattScaler":
        '''Fit Platt scaler by logistic regression on logit-transformed probabilities.

        Args:
            y_cal: Binary true labels (0/1) on calibration set, shape (n,).
            p_cal: Raw predicted probabilities P(Y=1), shape (n,).

        Returns:
            Self (fitted scaler).
        '''
        p_clipped  = np.clip(p_cal, 1e-7, 1.0 - 1e-7)
        logits_cal = (np.log(p_clipped) - np.log(1.0 - p_clipped)).reshape(-1, 1)
        self._logreg.fit(logits_cal, y_cal)
        self.a = float(self._logreg.coef_[0, 0])
        self.b = float(self._logreg.intercept_[0])
        return self

    def calibrate(self, p_raw: np.ndarray) -> np.ndarray:
        '''Apply Platt calibration to raw predicted probabilities.

        Args:
            p_raw: Raw predicted probabilities, shape (n,).

        Returns:
            Calibrated probabilities, shape (n,).
        '''
        p_clipped = np.clip(p_raw, 1e-7, 1.0 - 1e-7)
        logits    = (np.log(p_clipped) - np.log(1.0 - p_clipped)).reshape(-1, 1)
        return self._logreg.predict_proba(logits)[:, 1]


print("PlattScaler defined.")
ps_demo = PlattScaler().fit(y_cal, p_raw_cal)
print(f"  Platt fit: a={ps_demo.a:.4f}, b={ps_demo.b:.4f}")
print(f"  Interpretation: a > 1 -> further squeeze; a < 1 -> further spread")
print(f"  b != 0 -> systematic shift of the calibration curve")


### 1.4 Isotonic Regression Calibration

**Isotonic regression** learns a non-decreasing step function $f: [0,1] \to [0,1]$ that
minimises squared error subject to monotonicity:
$$
\min_{f \text{ non-decreasing}} \sum_{i=1}^n (f(\hat{p}_i) - y_i)^2
$$

Solved in $O(n)$ by the **pool adjacent violators (PAV)** algorithm.

**Trade-off vs. temperature/Platt:** isotonic is highly flexible (non-parametric) but
prone to overfitting on small calibration sets.  Best when $n_{\text{cal}} \gtrsim 1000$.


In [ ]:
# ── Part 1: Isotonic Regression Calibration ────────────────────────────────────


class IsotonicCalibrator:
    '''Non-parametric monotone calibration via isotonic (PAV) regression.

    Fits a non-decreasing step function from predicted probabilities to empirical
    accuracies using sklearn's IsotonicRegression.

    Attributes:
        _iso: Fitted sklearn IsotonicRegression instance.
    '''

    def __init__(self) -> None:
        '''Initialise IsotonicCalibrator with PAV-based isotonic regression.
                Uses out_of_bounds=clip to handle test probabilities outside training range.
        '''
        self._iso: SklearnIsotonic = SklearnIsotonic(
            out_of_bounds="clip", y_min=0.0, y_max=1.0,
        )

    def fit(
        self,
        y_cal: np.ndarray,
        p_cal: np.ndarray,
    ) -> "IsotonicCalibrator":
        '''Fit monotone calibration function.

        Args:
            y_cal: Binary true labels (0/1), shape (n,).
            p_cal: Raw predicted probabilities, shape (n,).

        Returns:
            Self (fitted calibrator).
        '''
        self._iso.fit(p_cal, y_cal.astype(float))
        return self

    def calibrate(self, p_raw: np.ndarray) -> np.ndarray:
        '''Apply isotonic calibration to raw probabilities.

        Args:
            p_raw: Raw predicted probabilities, shape (n,).

        Returns:
            Calibrated probabilities, shape (n,).
        '''
        return self._iso.predict(p_raw)


print("IsotonicCalibrator defined.")
iso_demo = IsotonicCalibrator().fit(y_cal, p_raw_cal)
n_knots = len(iso_demo._iso.X_thresholds_)
print(f"  Isotonic fit: {n_knots} knot positions learned from {len(y_cal)} calibration points")
print(f"  Input range: [{iso_demo._iso.X_min_:.4f}, {iso_demo._iso.X_max_:.4f}]")


### 1.5 Conformal Prediction

**Conformal prediction** provides **distribution-free** prediction *sets* with a
guaranteed marginal coverage property:
$$
P(Y_{n+1} \in \mathcal{C}(X_{n+1})) \geq 1 - \alpha
$$
for any finite $n$, under the sole assumption that data is **exchangeable** (e.g., i.i.d.).
No assumptions about the underlying model or distribution are required.

**Softmax Response Conformal Prediction (SRCP):**
1. **Nonconformity score:** $s_i = 1 - \hat{p}(Y_i \mid X_i)$ — surprise at true label.
2. **Threshold:** $\hat{q} = $ the $\lceil(n+1)(1-\alpha)\rceil/n$ quantile of $\{s_i\}_{i=1}^n$.
3. **Prediction set:** $\mathcal{C}(x) = \{y : 1 - \hat{p}(y \mid x) \leq \hat{q}\}$
   equivalently $\{y : \hat{p}(y \mid x) \geq 1 - \hat{q}\}$.

**Adaptive size:** uncertain examples get *larger* prediction sets; confident ones get singletons.


In [ ]:
# ── Part 1: Conformal Prediction ───────────────────────────────────────────────


class ConformalClassifier:
    '''Distribution-free prediction sets via softmax response conformal prediction.

    Computes sets C(x) = {y : p_hat(y|x) >= 1 - q_hat} where q_hat is the
    ceil((n+1)*(1-alpha))/n quantile of calibration nonconformity scores.

    Coverage guarantee: P(y_true in C(X)) >= 1 - alpha for exchangeable data.

    Attributes:
        alpha: Target miscoverage rate.
        threshold: Computed nonconformity threshold after calibration.
        n_cal: Number of calibration samples used.
        n_classes: Number of classes.
    '''

    def __init__(self, alpha: float = 0.10) -> None:
        '''Initialise ConformalClassifier with given significance level.
                Args:
        alpha: Target miscoverage rate. Prediction sets cover 1-alpha fraction.
        '''
        self.alpha:     float = alpha
        self.threshold: float = float("nan")
        self.n_cal:     int   = 0
        self.n_classes: int   = 0

    def calibrate(
        self,
        y_cal: np.ndarray,
        p_cal: np.ndarray,
    ) -> "ConformalClassifier":
        '''Compute the nonconformity threshold from calibration data.

        Args:
            y_cal: Integer class labels on calibration set, shape (n_cal,).
            p_cal: Probability matrix on calibration set, shape (n_cal, n_classes).

        Returns:
            Self (fitted conformal predictor).
        '''
        n              = len(y_cal)
        self.n_cal     = n
        self.n_classes = p_cal.shape[1]
        scores         = 1.0 - p_cal[np.arange(n), y_cal]
        level          = min(1.0, math.ceil((n + 1) * (1.0 - self.alpha)) / n)
        self.threshold = float(np.quantile(scores, level))
        return self

    def predict_set(
        self,
        p_test: np.ndarray,
    ) -> list[np.ndarray]:
        '''Compute prediction sets for test probability vectors.

        Args:
            p_test: Probability matrix on test set, shape (n_test, n_classes).

        Returns:
            List of n_test arrays; each contains the included class indices.
        '''
        sets: list[np.ndarray] = []
        for p_row in p_test:
            included = np.where(1.0 - p_row <= self.threshold)[0]
            if len(included) == 0:
                included = np.array([int(np.argmax(p_row))])  # fallback
            sets.append(included)
        return sets

    def empirical_coverage(
        self,
        y_test: np.ndarray,
        p_test: np.ndarray,
    ) -> float:
        '''Compute fraction of test samples where y_true is in prediction set.

        Args:
            y_test: True integer labels, shape (n_test,).
            p_test: Probability matrix, shape (n_test, n_classes).

        Returns:
            Empirical coverage in [0, 1].
        '''
        pred_sets = self.predict_set(p_test)
        covered   = sum(int(y_test[i] in ps) for i, ps in enumerate(pred_sets))
        return float(covered) / max(len(y_test), 1)

    def mean_set_size(self, p_test: np.ndarray) -> float:
        '''Compute average number of classes per prediction set.

        Args:
            p_test: Probability matrix, shape (n_test, n_classes).

        Returns:
            Mean set size as float.
        '''
        pred_sets = self.predict_set(p_test)
        return float(np.mean([len(ps) for ps in pred_sets]))


print("ConformalClassifier defined.")
# Quick demo on binary synthetic data
cp_demo = ConformalClassifier(alpha=ALPHA_CP)
cp_demo.calibrate(y_cal, p_raw_cal_2d)
print(f"  Conformal threshold q_hat = {cp_demo.threshold:.5f}")
print(f"  Guarantee: P(y_true in C(X)) >= {1-ALPHA_CP:.0%}")
print(f"  Threshold means: include y if P(y|x) >= {1-cp_demo.threshold:.5f}")


---
## Part 2 — Empirical Validation

We now apply all calibration methods to the synthetic binary dataset and compare:
1. Side-by-side reliability diagrams — visual calibration quality.
2. ECE and MCE table — quantitative comparison.
3. Calibration set size sensitivity — isotonic vs. parametric methods.


In [ ]:
# ── Apply All Calibration Methods to Synthetic Binary Data ────────────────────
# Fit on calibration set, evaluate on held-out test set

ts_synth  = TemperatureScaler(multiclass=False).fit(y_cal, p_raw_cal)
ps_synth  = PlattScaler().fit(y_cal, p_raw_cal)
iso_synth = IsotonicCalibrator().fit(y_cal, p_raw_cal)

p_ts_te   = ts_synth.calibrate(p_raw_te)
p_ps_te   = ps_synth.calibrate(p_raw_te)
p_iso_te  = iso_synth.calibrate(p_raw_te)

ece_raw_v, mce_raw_v = expected_calibration_error(y_te, p_raw_te,  N_BINS)
ece_ts_v,  mce_ts_v  = expected_calibration_error(y_te, p_ts_te,   N_BINS)
ece_ps_v,  mce_ps_v  = expected_calibration_error(y_te, p_ps_te,   N_BINS)
ece_iso_v, mce_iso_v = expected_calibration_error(y_te, p_iso_te,  N_BINS)

lbl_raw = "Uncalibrated"
lbl_ts  = f"Temp (T={ts_synth.T:.3f})"
lbl_ps  = f"Platt (a={ps_synth.a:.3f}, b={ps_synth.b:.3f})"
lbl_iso = "Isotonic"
print("Calibration comparison (synthetic binary, test set):")
print(f"  {'Method':>28s}  {'ECE':>8s}  {'MCE':>8s}  {'ECE improve':>12s}")
print("  " + "-" * 65)
for lbl, ece_v, mce_v in [
    (lbl_raw, ece_raw_v, mce_raw_v),
    (lbl_ts,  ece_ts_v,  mce_ts_v),
    (lbl_ps,  ece_ps_v,  mce_ps_v),
    (lbl_iso, ece_iso_v, mce_iso_v),
]:
    impr = (ece_raw_v - ece_v) / max(ece_raw_v, 1e-9) * 100.0
    print(f"  {lbl:>28s}  {ece_v:8.5f}  {mce_v:8.5f}  {impr:+11.1f}%")

# ── Side-by-side reliability diagrams ─────────────────────────────────────────
fig_cmp, axes_cmp = plt.subplots(2, 2, figsize=(13, 10))
ax_flat = axes_cmp.ravel()
colours = ["steelblue", "darkorange", "green", "purple"]
for ax_c, (lbl_c, p_c), col_c in zip(
    ax_flat,
    [(lbl_raw, p_raw_te), (lbl_ts, p_ts_te), (lbl_ps, p_ps_te), (lbl_iso, p_iso_te)],
    colours,
):
    ece_c, _ = expected_calibration_error(y_te, p_c, N_BINS)
    diag_c   = reliability_diagram_data(y_te, p_c, N_BINS)
    plot_reliability_diagram(diag_c, ax_c, title=lbl_c, bar_color=col_c, ece=ece_c)

plt.suptitle("Reliability Diagrams: Before and After Calibration",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()
print("All methods improve calibration; isotonic has smallest ECE but uses n_cal parameters.")


In [ ]:
# ── Calibration Set Size Sensitivity ──────────────────────────────────────────
# Key question: how does each method perform with fewer calibration samples?
# Prediction: isotonic overfits at small n; temperature/Platt are more stable.

cal_sizes = [20, 40, 80, 150, 300, 500]
rng_size  = np.random.default_rng(SEED + 7)
size_rows: list = []

for n_sz in cal_sizes:
    idx_sz   = rng_size.choice(len(y_cal), size=min(n_sz, len(y_cal)), replace=False)
    y_sz     = y_cal[idx_sz]
    p_sz     = p_raw_cal[idx_sz]

    ts_sz  = TemperatureScaler(multiclass=False).fit(y_sz, p_sz)
    ps_sz  = PlattScaler().fit(y_sz, p_sz)
    iso_sz = IsotonicCalibrator().fit(y_sz, p_sz)

    ece_ts_sz,  _ = expected_calibration_error(y_te, ts_sz.calibrate(p_raw_te),  N_BINS)
    ece_ps_sz,  _ = expected_calibration_error(y_te, ps_sz.calibrate(p_raw_te),  N_BINS)
    ece_iso_sz, _ = expected_calibration_error(y_te, iso_sz.calibrate(p_raw_te), N_BINS)
    size_rows.append((n_sz, ece_ts_sz, ece_ps_sz, ece_iso_sz))

print("ECE vs calibration set size:")
print(f"  {'n_cal':>6s}  {'Temp':>8s}  {'Platt':>8s}  {'Isotonic':>10s}")
print("  " + "-" * 38)
for n_sz, e_ts, e_ps, e_iso in size_rows:
    print(f"  {n_sz:6d}  {e_ts:8.5f}  {e_ps:8.5f}  {e_iso:10.5f}")
print("")
print(f"ECE with full calibration set (n={len(y_cal)}):")
print(f"  Temp={ece_ts_v:.5f}  Platt={ece_ps_v:.5f}  Isotonic={ece_iso_v:.5f}")

# Visualise size sensitivity
n_szs    = [r[0] for r in size_rows]
ece_tss  = [r[1] for r in size_rows]
ece_pss  = [r[2] for r in size_rows]
ece_isos = [r[3] for r in size_rows]

fig_sz, ax_sz = plt.subplots(figsize=(9, 5))
ax_sz.plot(n_szs, ece_tss,  "bo-", lw=2, markersize=7, label="Temperature scaling")
ax_sz.plot(n_szs, ece_pss,  "gs-", lw=2, markersize=7, label="Platt scaling")
ax_sz.plot(n_szs, ece_isos, "r^-", lw=2, markersize=7, label="Isotonic regression")
ax_sz.axhline(ece_raw_v, color="gray", ls=":", lw=1.5, label="Uncalibrated baseline")
ax_sz.set_xlabel("Calibration set size $n_{cal}$", fontsize=11)
ax_sz.set_ylabel("ECE on test set", fontsize=11)
ax_sz.set_title("Calibration Quality vs. Calibration Set Size",
                fontsize=11, fontweight="bold")
ax_sz.legend(fontsize=9)
ax_sz.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("Isotonic improves quickly with more data; parametric methods stable from ~50 samples.")
# Find the minimum n_cal for each method to beat uncalibrated ECE by 50%
target_ece = ece_raw_v * 0.5
crossover = {}
for method_k, ece_list_k in [('temp', ece_tss), ('platt', ece_pss), ('iso', ece_isos)]:
    for sz_k, ece_k in zip(n_szs, ece_list_k):
        if ece_k <= target_ece:
            crossover[method_k] = sz_k
            break
    else:
        crossover[method_k] = None
print(f"Min n_cal to achieve 50% ECE reduction (target ECE <= {target_ece:.5f}):")
for m_k, n_k in crossover.items():
    print(f"  {m_k:>8s}: n_cal = {n_k}")
print(f'Uncalibrated baseline ECE: {ece_raw_v:.5f}')


---
## Part 3 — Application to Real Data: Digits Dataset

We now apply calibration and conformal prediction to the **sklearn Digits** dataset
(10 handwritten digit classes, 8x8 pixel images, 1797 samples).  We use the same
Gaussian Naive Bayes, which is known to produce overconfident multi-class probabilities.

**Multi-class temperature scaling** uses a single $T$ shared across all classes.
**Conformal prediction** requires no calibration model — only a held-out set for
threshold estimation.


In [ ]:
# ── Load Digits Dataset (10-class classification) ─────────────────────────────
digits      = load_digits()
X_dig, y_dig = digits.data, digits.target
n_classes_dig = len(np.unique(y_dig))

# Three-way split: 60% train / 20% calibration / 20% test
X_tr_d, X_rest_d, y_tr_d, y_rest_d = train_test_split(
    X_dig, y_dig, test_size=0.40, random_state=SEED, stratify=y_dig,
)
X_cal_d, X_te_d, y_cal_d, y_te_d = train_test_split(
    X_rest_d, y_rest_d, test_size=0.50, random_state=SEED + 1, stratify=y_rest_d,
)

# Train GNB on Digits
gnb_dig = GaussianNB()
gnb_dig.fit(X_tr_d, y_tr_d)

p_raw_cal_d = gnb_dig.predict_proba(X_cal_d)   # (n_cal, 10)
p_raw_te_d  = gnb_dig.predict_proba(X_te_d)    # (n_te, 10)

acc_dig     = float(accuracy_score(y_te_d, gnb_dig.predict(X_te_d)))
ece_raw_d   = multi_class_ece(y_te_d, p_raw_te_d, N_BINS)

print("Digits dataset (10-class):")
print(f"  Train: {len(X_tr_d):4d} | Cal: {len(X_cal_d):4d} | Test: {len(X_te_d):4d}")
print(f"  Classes: {n_classes_dig}  (digits 0-9)")
print(f"  GNB test accuracy          : {acc_dig:.4f}")
print(f"  GNB ECE (multi-class conf) : {ece_raw_d:.5f}")
print("")
# Show top-class probability distribution
max_conf_d = p_raw_te_d.max(axis=1)
print(f"  Top-class probability: mean={max_conf_d.mean():.4f}  std={max_conf_d.std():.4f}")
print(f"  Fraction p_max > 0.99: {(max_conf_d > 0.99).mean():.3f}  (extreme overconfidence)")


### 3.1 Temperature Scaling on Digits

We fit a single temperature $T$ on the Digits calibration set.  The multi-class
version scales the log-probability vector element-wise and renormalises via softmax.


In [ ]:
# ── Temperature Scaling on Digits (Multi-Class) ────────────────────────────────
ts_dig  = TemperatureScaler(multiclass=True).fit(y_cal_d, p_raw_cal_d)
p_ts_d  = ts_dig.calibrate(p_raw_te_d)
ece_ts_d = multi_class_ece(y_te_d, p_ts_d, N_BINS)

print(f"Multi-class temperature scaling on Digits:")
print(f"  Optimal T*           = {ts_dig.T:.4f}")
print(f"  ECE before calibration: {ece_raw_d:.5f}")
print(f"  ECE after temp scaling: {ece_ts_d:.5f}")
impr_d = (ece_raw_d - ece_ts_d) / max(ece_raw_d, 1e-9) * 100.0
print(f"  ECE improvement        : {impr_d:.1f}%")
print("")

# Per-class reliability diagrams (2x5 grid, one per digit)
fig_dig, axes_dig = plt.subplots(2, 5, figsize=(18, 7))
for digit_k in range(10):
    ax_k    = axes_dig[digit_k // 5, digit_k % 5]
    y_ovr_k = (y_te_d == digit_k).astype(int)          # one-vs-rest label
    p_raw_k = p_raw_te_d[:, digit_k]
    p_ts_k  = p_ts_d[:, digit_k]

    ece_raw_k, _ = expected_calibration_error(y_ovr_k, p_raw_k, N_BINS)
    ece_ts_k, _  = expected_calibration_error(y_ovr_k, p_ts_k,  N_BINS)

    diag_ts_k = reliability_diagram_data(y_ovr_k, p_ts_k, N_BINS)
    diag_raw_k = reliability_diagram_data(y_ovr_k, p_raw_k, N_BINS)

    centers_k = diag_raw_k["bin_centers"]
    w_k       = centers_k[1] - centers_k[0] if len(centers_k) > 1 else 0.1

    ax_k.plot([0, 1], [0, 1], "k--", lw=1.2)
    for i_b in range(N_BINS):
        if diag_raw_k["counts"][i_b] > 0:
            ax_k.bar(centers_k[i_b], diag_raw_k["accuracies"][i_b],
                     width=w_k * 0.45, align="center", color="steelblue",
                     alpha=0.6, edgecolor="white", lw=0.3)
        if diag_ts_k["counts"][i_b] > 0:
            ax_k.bar(centers_k[i_b] + w_k * 0.45, diag_ts_k["accuracies"][i_b],
                     width=w_k * 0.45, align="center", color="darkorange",
                     alpha=0.6, edgecolor="white", lw=0.3)

    ax_k.set_title(f"Digit {digit_k} | raw ECE={ece_raw_k:.3f} ts={ece_ts_k:.3f}",
                   fontsize=8, fontweight="bold")
    ax_k.set_xlim(0, 1); ax_k.set_ylim(0, 1)
    ax_k.tick_params(labelsize=7)

plt.suptitle("Per-Digit Reliability: Blue=Uncalibrated, Orange=Temp-Scaled",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()
print("Temperature scaling improves calibration for most digits.")


### 3.2 Conformal Prediction Sets on Digits

Conformal prediction gives a **set** $\mathcal{C}(x)$ guaranteed to contain the true
digit with probability $\geq 1 - \alpha = 90\%$.  On easy examples the set is a
singleton; on hard examples it contains multiple plausible digits.


In [ ]:
# ── Conformal Prediction on Digits ────────────────────────────────────────────
cp_dig = ConformalClassifier(alpha=ALPHA_CP)
cp_dig.calibrate(y_cal_d, p_raw_cal_d)

coverage_dig  = cp_dig.empirical_coverage(y_te_d, p_raw_te_d)
mean_size_dig = cp_dig.mean_set_size(p_raw_te_d)

print(f"Conformal prediction on Digits (alpha={ALPHA_CP}, target>={1-ALPHA_CP:.0%} coverage):")
print(f"  Threshold q_hat    = {cp_dig.threshold:.5f}")
print(f"  Empirical coverage = {coverage_dig:.4f}  (guaranteed >= {1-ALPHA_CP:.2f})")
print(f"  Mean set size      = {mean_size_dig:.3f}")
print(f"  Coverage satisfied : {'YES' if coverage_dig >= 1 - ALPHA_CP else 'NO'}")
print("")

# Show prediction sets for a sample of test points
pred_sets_dig = cp_dig.predict_set(p_raw_te_d)
print("Sample prediction sets (first 15 test digits):")
header_idx = "Idx"; header_true = "True"; header_set = "Prediction set"; header_cov = "Covered?"
print(f"  {header_idx:>4s}  {header_true:>5s}  {header_set:<28s}  {header_cov}")
print("  " + "-" * 50)
for i in range(15):
    ps_str   = "{" + ",".join(str(c) for c in pred_sets_dig[i]) + "}"
    covered  = "YES" if y_te_d[i] in pred_sets_dig[i] else "NO"
    print(f"  {i:4d}  {y_te_d[i]:5d}  {ps_str:<28s}  {covered}")

# Distribution of set sizes
set_sizes_dig = np.array([len(ps) for ps in pred_sets_dig])
size_dist     = {k: int((set_sizes_dig == k).sum()) for k in range(1, 11) if (set_sizes_dig == k).sum() > 0}
print("")
print("Set size distribution:")
for sz_k, cnt_k in sorted(size_dist.items()):
    bar_k = "#" * (cnt_k // 3)
    print(f"  Size {sz_k:2d}: {cnt_k:4d}  {bar_k}")


### 2.2 Probability Distribution Shift

Calibration methods differ in *how* they redistribute probability mass.
Temperature scaling applies a symmetric squeeze/stretch; isotonic regression
can make arbitrary monotone adjustments.


In [ ]:
# ── Probability Shift Analysis: Before vs After Calibration ───────────────────
# Shows how each method redistributes probability mass for binary synthetic data.

prob_bins  = np.linspace(0, 1, 25)
bin_c_vis  = 0.5 * (prob_bins[:-1] + prob_bins[1:])

# Count samples in each probability bin for each method
def hist_probs(p_vals: np.ndarray, bins: np.ndarray) -> np.ndarray:
    '''Count samples in each probability bin.

    Args:
        p_vals: Probability values, shape (n,).
        bins: Bin edges, shape (n_bins+1,).

    Returns:
        Counts per bin, shape (n_bins,).
    '''
    counts = np.zeros(len(bins) - 1, dtype=int)
    for b in range(len(bins) - 1):
        if b < len(bins) - 2:
            counts[b] = int(np.sum((p_vals >= bins[b]) & (p_vals < bins[b + 1])))
        else:
            counts[b] = int(np.sum((p_vals >= bins[b]) & (p_vals <= bins[b + 1])))
    return counts


counts_raw = hist_probs(p_raw_te,  prob_bins)
counts_ts  = hist_probs(p_ts_te,   prob_bins)
counts_ps  = hist_probs(p_ps_te,   prob_bins)
counts_iso = hist_probs(p_iso_te,  prob_bins)

fig_ps, axes_ps = plt.subplots(2, 2, figsize=(13, 8))
ax_flat_ps = axes_ps.ravel()
data_ps = [
    ("Uncalibrated (GNB)", counts_raw, "steelblue"),
    (lbl_ts,               counts_ts,  "darkorange"),
    (lbl_ps,               counts_ps,  "green"),
    (lbl_iso,              counts_iso, "purple"),
]
for ax_p, (lbl_p, cnt_p, col_p) in zip(ax_flat_ps, data_ps):
    ax_p.bar(bin_c_vis, cnt_p / max(cnt_p.sum(), 1), width=0.03,
             color=col_p, alpha=0.75, edgecolor="white")
    ax_p.axvline(0.5, color="black", ls="--", lw=1.2)
    ax_p.set_xlabel("Predicted probability")
    ax_p.set_ylabel("Fraction of samples")
    ax_p.set_title(lbl_p, fontsize=10, fontweight="bold")
    ax_p.set_xlim(0, 1)

plt.suptitle("Probability Mass Redistribution by Calibration Method",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

# Quantitative: how much mass is moved away from extremes (p < 0.05 or p > 0.95)?
def extreme_mass(p_vals: np.ndarray, thr: float = 0.05) -> float:
    '''Compute fraction of probability mass in the extreme regions [0, thr] u [1-thr, 1].

    Args:
        p_vals: Probability values, shape (n,).
        thr: Threshold defining extreme region.

    Returns:
        Fraction of samples in extreme region.
    '''
    return float(np.mean((p_vals < thr) | (p_vals > 1.0 - thr)))


print("Fraction of predictions in extreme regions (p < 0.05 or p > 0.95):")
for lbl_e, p_e in [(lbl_raw, p_raw_te), (lbl_ts, p_ts_te),
                    (lbl_ps, p_ps_te), (lbl_iso, p_iso_te)]:
    em = extreme_mass(p_e)
    bar_e = "#" * int(em * 50)
    print(f"  {lbl_e:>28s}: {em:.4f}  |{bar_e}")
print("Calibration redistributes extreme predictions toward moderate confidence.")


---
## Part 4 — Theory vs Practice Analysis

We compare calibration methods against theoretical predictions:
1. **ECE table:** does calibration consistently reduce ECE?
2. **Isotonic overfitting:** theory says isotonic needs large $n_{\text{cal}}$.
3. **Conformal coverage:** the theoretical guarantee says empirical coverage $\geq 1-\alpha$.
   We verify this holds across multiple $\alpha$ levels.


In [ ]:
# ── Full Calibration Comparison Table ─────────────────────────────────────────
# Binary (synthetic) and multi-class (Digits) comparisons

results_bin = pd.DataFrame({
    "Method":    [lbl_raw, lbl_ts, lbl_ps, lbl_iso],
    "ECE":       [ece_raw_v, ece_ts_v, ece_ps_v, ece_iso_v],
    "MCE":       [mce_raw_v, mce_ts_v, mce_ps_v, mce_iso_v],
    "Params":    [0, 1, 2, len(y_cal)],
    "ECE Improv":[(ece_raw_v - e) / max(ece_raw_v, 1e-9) * 100
                  for e in [ece_raw_v, ece_ts_v, ece_ps_v, ece_iso_v]],
})
results_bin = results_bin.sort_values("ECE")

results_mc = pd.DataFrame({
    "Method":   ["GNB (uncalibrated)", f"Temp scaling (T={ts_dig.T:.3f})"],
    "ECE (MC)": [ece_raw_d, ece_ts_d],
    "Accuracy": [acc_dig, float(accuracy_score(y_te_d,
                  np.argmax(p_ts_d, axis=1)))],
})

print("Binary Calibration Comparison (synthetic, test set):")
print(results_bin.to_string(index=False, float_format=lambda x: f"{x:.5f}"))
print("")
print("Multi-class Calibration Comparison (Digits, test set):")
print(results_mc.to_string(index=False, float_format=lambda x: f"{x:.5f}"))
print("")
print("Observations:")
print("  1. All methods reduce ECE vs. uncalibrated baseline.")
print("  2. Isotonic achieves lowest ECE but uses n_cal parameters.")
print("  3. Temperature scaling is most parameter-efficient (1 param vs. n_cal).")
print("  4. Calibration does NOT change predictions -> accuracy unchanged.")


# ── NLL comparison: how much does calibration improve log-likelihood? ─────────
def binary_nll(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    '''Compute binary negative log-likelihood.

    Args:
        y_true: Binary labels, shape (n,).
        y_prob: Predicted probabilities, shape (n,).

    Returns:
        Mean NLL as float.
    '''
    p = np.clip(y_prob, 1e-7, 1.0 - 1e-7)
    return -float(np.mean(y_true * np.log(p) + (1.0 - y_true) * np.log(1.0 - p)))


nll_raw_v = binary_nll(y_te, p_raw_te)
nll_ts_v  = binary_nll(y_te, p_ts_te)
nll_ps_v  = binary_nll(y_te, p_ps_te)
nll_iso_v = binary_nll(y_te, p_iso_te)
nll_opt   = binary_nll(y_te, y_te.astype(float))  # lower bound (perfect)

print("")
print("Negative log-likelihood comparison (lower = better):")
hdr_m = "Method"; hdr_nll = "NLL"; hdr_pct = "% of optimal"
print(f"  {hdr_m:>28s}  {hdr_nll:>8s}  {hdr_pct:>12s}")
print("  " + "-" * 55)
for lbl_n, nll_n in [(lbl_raw, nll_raw_v), (lbl_ts, nll_ts_v),
                      (lbl_ps, nll_ps_v), (lbl_iso, nll_iso_v)]:
    pct_of_opt = nll_n / max(abs(nll_opt), 1e-9) * 100.0
    print(f"  {lbl_n:>28s}  {nll_n:8.5f}  {pct_of_opt:12.2f}%")
print(f"  {'Perfect predictor':>28s}  {nll_opt:8.5f}  {'100.00%':>12s}")
print("")
print("Calibration reduces NLL because corrected probabilities waste less log-probability.")
print("Temperature scaling: scales logits to match the calibration set's label frequency.")


In [ ]:
# ── Conformal Prediction Coverage Sweep: Theory vs. Empirical ─────────────────
# Theory: for any alpha, P(y_true in C(X)) >= 1 - alpha
# We verify this across 10 alpha levels on both binary and multi-class data.

alpha_levels = [0.01, 0.03, 0.05, 0.08, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40]
sweep_rows: list = []

for alpha_v in alpha_levels:
    # Binary (synthetic)
    cp_bin = ConformalClassifier(alpha=alpha_v)
    cp_bin.calibrate(y_cal, p_raw_cal_2d)
    cov_bin  = cp_bin.empirical_coverage(y_te, p_raw_te_2d)
    size_bin = cp_bin.mean_set_size(p_raw_te_2d)
    # Multi-class (Digits)
    cp_mc = ConformalClassifier(alpha=alpha_v)
    cp_mc.calibrate(y_cal_d, p_raw_cal_d)
    cov_mc  = cp_mc.empirical_coverage(y_te_d, p_raw_te_d)
    size_mc = cp_mc.mean_set_size(p_raw_te_d)
    sweep_rows.append((alpha_v, 1 - alpha_v, cov_bin, size_bin,
                        cov_mc, size_mc,
                        "YES" if cov_bin >= 1 - alpha_v else "NO",
                        "YES" if cov_mc  >= 1 - alpha_v else "NO"))

hdr_alpha = "alpha"; hdr_target = "Target"; hdr_cov_b = "Cov(bin)"; hdr_sz_b = "Size(bin)"
hdr_cov_d = "Cov(dig)"; hdr_sz_d = "Size(dig)"; hdr_ok_b = "OK(b)"; hdr_ok_d = "OK(d)"
print("Conformal prediction coverage verification:")
print(f"  {hdr_alpha:>6s}  {hdr_target:>7s}  {hdr_cov_b:>9s}  {hdr_sz_b:>9s}  "
      f"{hdr_cov_d:>9s}  {hdr_sz_d:>9s}  {hdr_ok_b:>5s}  {hdr_ok_d:>5s}")
print("  " + "-" * 75)
for r in sweep_rows:
    print(f"  {r[0]:6.2f}  {r[1]:7.2f}  {r[2]:9.4f}  {r[3]:9.3f}  "
          f"{r[4]:9.4f}  {r[5]:9.3f}  {r[6]:>5s}  {r[7]:>5s}")

n_ok_bin = sum(1 for r in sweep_rows if r[6] == "YES")
n_ok_dig = sum(1 for r in sweep_rows if r[7] == "YES")
print("")
print(f"Guarantee satisfied: binary={n_ok_bin}/{len(alpha_levels)}, Digits={n_ok_dig}/{len(alpha_levels)}")
print("  Coverage always >= 1-alpha (finite-sample guarantee holds empirically).")
print("  Set size grows as alpha -> 0 (to ensure coverage on harder examples).")

# Visualise
fig_cv, axes_cv = plt.subplots(1, 2, figsize=(13, 5))
target_cov = [1 - r[0] for r in sweep_rows]
cov_bins   = [r[2] for r in sweep_rows]
cov_digs   = [r[4] for r in sweep_rows]
sz_bins    = [r[3] for r in sweep_rows]
sz_digs    = [r[5] for r in sweep_rows]

axes_cv[0].plot(alpha_levels, target_cov, "k--", lw=2, label="Target 1-alpha")
axes_cv[0].plot(alpha_levels, cov_bins,   "bo-", lw=2, markersize=7, label="Binary (empirical)")
axes_cv[0].plot(alpha_levels, cov_digs,   "rs-", lw=2, markersize=7, label="Digits (empirical)")
axes_cv[0].fill_between(alpha_levels, target_cov, [1.0]*len(alpha_levels),
                         alpha=0.1, color="green", label="Valid region")
axes_cv[0].set_xlabel("alpha (target miscoverage rate)", fontsize=11)
axes_cv[0].set_ylabel("Empirical coverage", fontsize=11)
axes_cv[0].set_title("Coverage Guarantee: Empirical >= 1-alpha", fontsize=11, fontweight="bold")
axes_cv[0].legend(fontsize=9)

axes_cv[1].plot(alpha_levels, sz_bins, "bo-", lw=2, markersize=7, label="Binary (n_classes=2)")
axes_cv[1].plot(alpha_levels, sz_digs, "rs-", lw=2, markersize=7, label="Digits (n_classes=10)")
axes_cv[1].set_xlabel("alpha (target miscoverage rate)", fontsize=11)
axes_cv[1].set_ylabel("Mean set size", fontsize=11)
axes_cv[1].set_title("Mean Prediction Set Size vs. alpha",
                      fontsize=11, fontweight="bold")
axes_cv[1].legend(fontsize=9)

plt.suptitle("Conformal Prediction: Coverage Guarantee Verification",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()


### 4.2 Conformal Set Adaptivity

Conformal prediction sets should be **adaptive**: small on easy examples
(high top-class confidence), large on ambiguous ones.  We verify this
and show that hard examples (mis-predicted) receive larger sets.


In [ ]:
# ── Conformal Prediction: Adaptive Set Size Analysis ──────────────────────────
# Conformal sets adapt to difficulty — easy examples get singletons, hard ones
# get larger sets.  We analyse this adaptivity on the Digits dataset.

pred_sets_full = cp_dig.predict_set(p_raw_te_d)
set_sizes_full = np.array([len(ps) for ps in pred_sets_full])

# Separate by whether prediction was correct
correct_mask = np.array([y_te_d[i] in pred_sets_full[i] for i in range(len(y_te_d))])
incorrect    = ~correct_mask
covered_all  = correct_mask.mean()

# Top predicted probability vs set size
top_probs    = p_raw_te_d.max(axis=1)
top_preds    = p_raw_te_d.argmax(axis=1)
is_correct_pred = (top_preds == y_te_d)

print(f"Conformal set size analysis (Digits, alpha={ALPHA_CP}):")
print(f"  Overall coverage: {covered_all:.4f}  (target >= {1-ALPHA_CP:.2f})")
print("")
print("  Set size distribution:")
for sz_v in range(1, 11):
    n_sz_v = int((set_sizes_full == sz_v).sum())
    if n_sz_v > 0:
        bar_v = "#" * (n_sz_v // 5)
        print(f"    Size {sz_v:2d}: {n_sz_v:4d}  {bar_v}")
print("")
print("  Mean set size per top-1 confidence decile:")
decile_edges = np.percentile(top_probs, np.arange(0, 110, 10))
for dec in range(10):
    mask_dec = (top_probs >= decile_edges[dec]) & (top_probs < decile_edges[dec + 1])
    if mask_dec.sum() > 0:
        mean_sz_dec = set_sizes_full[mask_dec].mean()
        acc_dec     = is_correct_pred[mask_dec].mean()
        print(f"    Decile {dec+1:2d} (conf [{decile_edges[dec]:.2f},{decile_edges[dec+1]:.2f}]): "
              f"mean_set_size={mean_sz_dec:.2f}  acc={acc_dec:.3f}")

# Visualise: top-class confidence vs set size
fig_ca, axes_ca = plt.subplots(1, 2, figsize=(13, 5))

axes_ca[0].scatter(top_probs[is_correct_pred],
                   set_sizes_full[is_correct_pred],
                   alpha=0.4, s=15, c="steelblue", label="Correct prediction")
axes_ca[0].scatter(top_probs[~is_correct_pred],
                   set_sizes_full[~is_correct_pred],
                   alpha=0.6, s=25, c="tomato", marker="x", label="Wrong prediction")
axes_ca[0].set_xlabel("Top-class confidence P(argmax|x)", fontsize=11)
axes_ca[0].set_ylabel("Conformal set size", fontsize=11)
axes_ca[0].set_title("Adaptivity: Confident -> Small Set | Uncertain -> Large Set",
                      fontsize=10, fontweight="bold")
axes_ca[0].legend(fontsize=9)

# Histogram of set sizes split by correctness
size_range = np.arange(1, set_sizes_full.max() + 2)
n_correct_by_size   = np.array([int(((set_sizes_full == sz) & is_correct_pred).sum()) for sz in size_range])
n_incorrect_by_size = np.array([int(((set_sizes_full == sz) & ~is_correct_pred).sum()) for sz in size_range])
axes_ca[1].bar(size_range, n_correct_by_size, color="steelblue", alpha=0.7, label="Correct top-1")
axes_ca[1].bar(size_range, n_incorrect_by_size, bottom=n_correct_by_size,
               color="tomato", alpha=0.7, label="Wrong top-1")
axes_ca[1].set_xlabel("Conformal prediction set size", fontsize=11)
axes_ca[1].set_ylabel("Count", fontsize=11)
axes_ca[1].set_title("Set Size Distribution (Digits)", fontsize=11, fontweight="bold")
axes_ca[1].legend(fontsize=9)

plt.suptitle("Conformal Prediction Set Analysis: Adaptivity to Difficulty",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()
print("Key insight: larger sets appear on misclassified examples.")
print("  Conformal prediction 'knows' it is unsure by giving you more candidates.")


---
## Part 5 — Summary & Lessons Learned

### Key Takeaways

1. **Calibration is distinct from accuracy.** A highly accurate model can still be
   poorly calibrated (overconfident or underconfident). ECE measures calibration quality
   independently of predictive power.

2. **Temperature scaling is the practical default.** Single parameter, optimizer-free
   (minimize NLL on cal set), preserves predictions, works for multi-class.  It is the
   same mechanism used to control LLM generation temperature in Modules 17–18.

3. **Isotonic regression is more flexible but data-hungry.** With large calibration sets
   it achieves lower ECE than parametric methods, but it can overfit with $n_{\text{cal}} < 100$.

4. **Conformal prediction provides a distribution-free guarantee.** Unlike calibration
   methods, conformal prediction does not require correct probability estimation — it
   directly guarantees $P(y_{\text{true}} \in \mathcal{C}(X)) \geq 1 - \alpha$ under
   exchangeability.  Set sizes adapt to difficulty.

5. **Calibration does not change predictions.** All post-hoc calibration methods
   (temperature, Platt, isotonic) preserve argmax — accuracy is unaffected.

### What's Next

→ **04-10 (Gaussian Processes & Bayesian Optimization)** extends uncertainty quantification
to the kernel-based GP framework, providing principled uncertainty bands over function space.

### Going Further

- Guo et al. (2017). *On Calibration of Modern Neural Networks.* ICML.
- Platt (1999). *Probabilistic Outputs for Support Vector Machines.*
- Venn-Abers predictors: an extension of isotonic regression with coverage guarantees.
- Angelopoulos & Bates (2023). *Conformal Risk Control.* ICLR.
  (Extends conformal prediction beyond coverage to risk-controlling sets.)
- Shafer & Vovk (2008). *A Tutorial on Conformal Prediction.* JMLR.


In [ ]:
# ── Final Summary ─────────────────────────────────────────────────────────────
print("=" * 72)
print("04-09 CALIBRATION & UNCERTAINTY QUANTIFICATION — SUMMARY")
print("=" * 72)
print("")
print("1. Synthetic binary (GNB, n_test=", len(y_te), ")")
print(f"   ECE: uncal={ece_raw_v:.5f} -> temp={ece_ts_v:.5f} / platt={ece_ps_v:.5f} / iso={ece_iso_v:.5f}")
print(f"   Best ECE: {min(ece_ts_v, ece_ps_v, ece_iso_v):.5f}")
print("")
print("2. Multi-class Digits (GNB, n_test=", len(y_te_d), ")")
print(f"   ECE: uncal={ece_raw_d:.5f} -> temp scaled={ece_ts_d:.5f}")
print(f"   Optimal T* = {ts_dig.T:.4f}")
print("")
print("3. Conformal prediction (binary, alpha=", ALPHA_CP, ")")
print(f"   Threshold q_hat = {cp_demo.threshold:.5f}")
print(f"   Coverage guarantee >= {1-ALPHA_CP:.2f} : satisfied={coverage_dig >= 1-ALPHA_CP}")
print("")
print("4. Coverage sweep: guarantee holds at ALL alpha levels")
print(f"   Binary : {n_ok_bin}/{len(alpha_levels)} levels satisfied")
print(f"   Digits : {n_ok_dig}/{len(alpha_levels)} levels satisfied")
print("")
print("=" * 72)
assert ece_ts_v  < ece_raw_v, "Temperature scaling must improve ECE"
assert ece_iso_v < ece_raw_v, "Isotonic must improve ECE"
assert coverage_dig >= 1 - ALPHA_CP, "Conformal coverage guarantee must hold on Digits"
assert ts_dig.T > 1.0, "GNB should be overconfident -> T* > 1"
print("All sanity assertions passed.")
# ── Extended assertions and cross-method analysis ──────────────────────────
assert ece_ps_v  < ece_raw_v, 'Platt scaling must improve ECE'
assert ece_ts_v  < ece_raw_v, 'Temperature scaling must improve ECE'
assert n_ok_bin == len(alpha_levels), 'All binary coverage levels must be satisfied'
assert n_ok_dig == len(alpha_levels), 'All Digits coverage levels must be satisfied'
assert ts_synth.T > 1.0, 'GNB overconfident -> optimal T > 1'
assert cp_demo.threshold > 0, 'Conformal threshold must be positive'
# NLL: calibration should reduce NLL
assert nll_ts_v  < nll_raw_v, 'Temperature scaling must reduce NLL'
assert nll_ps_v  < nll_raw_v, 'Platt scaling must reduce NLL'
assert nll_iso_v < nll_raw_v, 'Isotonic must reduce NLL'
best_ece_method = min([(ece_ts_v, lbl_ts), (ece_ps_v, lbl_ps), (ece_iso_v, lbl_iso)])
best_nll_method = min([(nll_ts_v, lbl_ts), (nll_ps_v, lbl_ps), (nll_iso_v, lbl_iso)])
print(f"Best ECE: {best_ece_method[1]} = {best_ece_method[0]:.5f}")
print(f"Best NLL: {best_nll_method[1]} = {best_nll_method[0]:.5f}")
print("All extended assertions passed.")
